# Imports

In [1]:
from petrobras_dataset import read_all_wells_with_dept_to_list, filter_commom_features
from utils.training_utilities import add_derived_features, WarmupScheduler, evaluate_model, set_deterministic
from utils.modelTrainer import load_best_configuration, FinalModelTrainer, train_model_with_validation_split
import os
from crossfold_hyperparamet_experiment import CrossFoldHyperparameterExperiment

set_deterministic(42)

# Main pipeline execution
Improved Well Log VS Prediction Pipeline
## Configuration - Improved

In [2]:
base_config = {
    "sequence_length": 15,
    "mask_value": -1.0,
    "num_epochs": 150,
    "patience": 30,
    "target_feature": "VS",
    "clusters": {
        "A": [2, 3, 4],
        "B": [0, 1]
    }
}

## Features Combinations - MORE FEATURES

In [3]:
print("=" * 80)
print("IMPROVED WELL LOG VS PREDICTION PIPELINE")
print("=" * 80)
    # Feature combinations - MORE FEATURES
feature_combinations = [
        # More features = better predictions
        ["VP", "RHO", "GR", "CALIPER", "POROSIDADE", "SATURACAO", "ARGILOSIDADE"],
        ["VP", "RHO", "GR", "CALIPER", "POROSIDADE", "SATURACAO"],
        # With derived features (if available)
        ["VP", "RHO", "ACOUSTIC_IMP", "GR", "POROSIDADE", "SATURACAO"],
        # Core physics-based
        ["VP", "RHO", "POROSIDADE", "GR", "ARGILOSIDADE"],
        ["VP", "RHO", "POROSIDADE", "SATURACAO"],
    ]

IMPROVED WELL LOG VS PREDICTION PIPELINE


## Step 1: Load Data

In [4]:
print("\n" + "=" * 80)
print("STEP 1: LOADING WELL DATA")
print("=" * 80)

wells = read_all_wells_with_dept_to_list(features="all")
well_dfs = filter_commom_features(wells, ignore=["VS"])


STEP 1: LOADING WELL DATA


### Add derived features

In [5]:
print("Adding derived features...")
well_dfs = [add_derived_features(df) for df in well_dfs]

wells_with_vs = [df for df in well_dfs if "VS" in df.columns]
wells_without_vs = [df for df in well_dfs if "VS" not in df.columns]

print(f"Total wells loaded: {len(well_dfs)}")
print(f"Wells with VS: {len(wells_with_vs)}")
print(f"Wells without VS: {len(wells_without_vs)}")

Adding derived features...
Total wells loaded: 7
Wells with VS: 5
Wells without VS: 2


## Step 2: Run Cross-Fold Validation

In [6]:
print("\n" + "=" * 80)
print("STEP 2: CROSS-FOLD VALIDATION")
print("=" * 80)

existing_experiments = [d for d in os.listdir("experiments") if d.startswith("experiment_")]

if existing_experiments:
    print(f"\nFound {len(existing_experiments)} existing experiment(s):")
    for i, exp_dir in enumerate(existing_experiments):
        print(f" {i + 1}. {exp_dir}")

    response = (
        input("\nDo you want to use existing experiments(s) (y/n): ").strip().lower()
    )

    if response == "y":
        if len(existing_experiments) == 1:
            experiment_dir = existing_experiments[0]
        else:
            exp_idx = (
                int(input(f"Which experiment? (1-{len(existing_experiments)}): ").strip()) - 1
            )
            experiment_dir = existing_experiments[exp_idx]
        
        print(f"\nUsing existing experiment: {experiment_dir}")
        best_config = load_best_configuration(experiment_dir)
    else:
        print("\nRunning new cross-fold validation...")
        experiment = CrossFoldHyperparameterExperiment(base_config)
        results = experiment.run_cross_fold_experiments(feature_combinations, wells_with_vs)
        experiment_dir = experiment.results_dir
        best_config = load_best_configuration(experiment_dir)
else:
    print("\nNo existing experiments found. Running cross-fold validation...")
    experiment = CrossFoldHyperparameterExperiment(base_config)
    results = experiment.run_cross_fold_experiments(feature_combinations, wells_with_vs)
    experiment_dir = experiment.results_dir
    best_config = load_best_configuration(experiment_dir)


STEP 2: CROSS-FOLD VALIDATION

Found 7 existing experiment(s):
 1. experiment_20260427_145352
 2. experiment_20260504_105147
 3. experiment_20260504_111728
 4. experiment_20260427_095356
 5. experiment_20260504_105350
 6. experiment_20260427_140813
 7. experiment_20260504_105229

Running new cross-fold validation...
Total wells with VS: 5
Running 1 hyperparameter configurations
Testing 5 feature combinations
Cross-fold validation with 5 wells
Total experiments: 25

FEATURE COMBINATION: ['VP', 'RHO', 'GR', 'CALIPER', 'POROSIDADE', 'SATURACAO', 'ARGILOSIDADE']

--------------------------------------------------------------------------------
CLUSTER A: Wells [2, 3, 4]
--------------------------------------------------------------------------------

--------------------------------------------------------------------------------
FOLD 1/3: Using Well 2 as test set
--------------------------------------------------------------------------------

Experiment 1/25
  Hyperparameters: {'embed_di

## STEP 3: Train Final Model

In [7]:
print("\n" + "=" * 80)
print("STEP 3: TRAINING FINAL MODEL")
print("=" * 80)

final_output_dir = os.path.join(experiment_dir, "final_model")
trainer = FinalModelTrainer(best_config, base_config, output_dir=final_output_dir)
trained_clusters = trainer.train_final_model(
    wells_with_vs, target_feature=base_config["target_feature"]
)


STEP 3: TRAINING FINAL MODEL

TRAINING FINAL MODELS PER CLUSTER

CLUSTER A — Wells [2, 3, 4]
  Features: ['VP', 'RHO', 'GR', 'CALIPER', 'POROSIDADE', 'SATURACAO', 'ARGILOSIDADE']
  Hyperparameters: {'embed_dim': 160, 'num_heads': 10, 'num_blocks': 3, 'dropout': 0.2, 'learning_rate': 0.0005, 'batch_size': 32, 'scheduler_type': 'plateau', 'criterion_type': 'huber', 'optimizer_type': 'adamw'}
  Total training samples: 4122
  Training sequences: 4107

  Training cluster A...
Epoch 1/150
Train loss: 0.0194 | Val loss: 0.0082 | Val R2: 0.1693
Epoch 11/150
Train loss: 0.0051 | Val loss: 0.0197 | Val R2: -0.9851
Epoch 21/150
Train loss: 0.0029 | Val loss: 0.0172 | Val R2: -0.7338
Epoch 31/150
Train loss: 0.0028 | Val loss: 0.0183 | Val R2: -0.8531
Early stopping at epoch 31
  Training complete!
  Best validation loss: 0.008240
  Final training R²: 0.7785
  Final validation R²: -0.8531
  Model saved to: experiments/experiment_20260504_114244/final_model/final_model_cluster_A.pth
  Scaler saved

## STEP 4: Make Predictions

In [8]:
print(f"\n" + "=" * 80)
print("STEP 4: MAKING PREDICTIONS")
print("=" * 80)

all_results = trainer.predict_on_wells(
    trained_clusters,
    wells_without_vs,
    wells_with_vs,
    target_feature=base_config["target_feature"]
)


STEP 4: MAKING PREDICTIONS

MAKING PREDICTIONS

Predicting on 2 wells WITHOUT VS:

  Well 1/2
  well_without_vs_0 → Cluster A
    Generated 1286 predictions
    VS range: [0.950, 1.439]

  Well 2/2
  well_without_vs_1 → Cluster B
    Generated 2596 predictions
    VS range: [0.955, 1.866]

Validating on 5 wells WITH VS:

  Well 1/5
  well_with_vs_0 → Cluster B
    Generated 986 predictions
    R²: 0.6895
    RMSE: 0.1114
    MSE: 0.0124
    MAE: 0.0945

  Well 2/5
  well_with_vs_1 → Cluster B
    Generated 1186 predictions
    R²: -0.0872
    RMSE: 0.0902
    MSE: 0.0081
    MAE: 0.0750

  Well 3/5
  well_with_vs_2 → Cluster A
    Generated 1136 predictions
    R²: 0.5568
    RMSE: 0.0784
    MSE: 0.0061
    MAE: 0.0608

  Well 4/5
  well_with_vs_3 → Cluster A
    Generated 1576 predictions
    R²: 0.8271
    RMSE: 0.0763
    MSE: 0.0058
    MAE: 0.0573

  Well 5/5
  well_with_vs_4 → Cluster A
    Generated 1365 predictions
    R²: 0.2682
    RMSE: 0.1239
    MSE: 0.0154
    MAE: 0.10

## STEP 5: Generate Plots and Reports

In [9]:
print("\n" + "=" * 80)
print("STEP 5: GENERATING REPORTS AND PLOTS")
print("=" * 80)

trainer.plot_predictions(all_results)
trainer.generate_summary_report(all_results, best_config)



STEP 5: GENERATING REPORTS AND PLOTS

GENERATING PLOTS
Predictions plot (wells without VS) saved to: experiments/experiment_20260504_114244/final_model/plots/predictions_wells_without_vs.png
Time series comparison plot saved to: experiments/experiment_20260504_114244/final_model/plots/predictions_vs_actuals_timeseries.png
Scatter plot saved to: experiments/experiment_20260504_114244/final_model/plots/predictions_vs_actuals_scatter.png
Error distribution plot saved to: experiments/experiment_20260504_114244/final_model/plots/error_distribution.png

Summary report saved to: experiments/experiment_20260504_114244/final_model/SUMMARY_REPORT.txt

WELL LOG VS PREDICTION - FINAL MODEL SUMMARY REPORT

Generated: 2026-05-04 12:03:31

--------------------------------------------------------------------------------
BEST MODEL CONFIGURATION
--------------------------------------------------------------------------------
Cluster: A
  Wells: [2, 3, 4]
  Features: ['VP', 'RHO', 'GR', 'CALIPER', 'POR

# Done

In [10]:
print("\n" + "=" * 80)
print("PIPELINE COMPLETE!")
print("=" * 80)
print(f"\nAll results saved to: {trainer.output_dir}/")
print(f"  - Final model: final_model.pth")
print(f"  - Scaler: final_scaler.pkl")
print(f"  - Predictions: all_predictions.json")
print(f"  - Summary: SUMMARY_REPORT.txt")
print(f"  - Plots: plots/")
print("\n" + "=" * 80)


PIPELINE COMPLETE!

All results saved to: experiments/experiment_20260504_114244/final_model/
  - Final model: final_model.pth
  - Scaler: final_scaler.pkl
  - Predictions: all_predictions.json
  - Summary: SUMMARY_REPORT.txt
  - Plots: plots/

